# Kikuchi Sphere Viewer

In [ ]:
import os

import imageio.v2 as imageio
import numpy as np
import pyvista as pv

pattern_name = "pattern1.bmp"
pattern_dir = "patterns"
pcx, pcy, pcz = 0.5, 0.5, 0.6
intensity_threshold = 0.05

pattern_path = os.path.join(pattern_dir, pattern_name)
if not os.path.exists(pattern_path):
    raise FileNotFoundError(f"Pattern file not found: {pattern_path}")

img = imageio.imread(pattern_path)
if img.ndim == 3:
    img = img[:, :, 0]

img = img.astype(float)
img = (img - img.min()) / (img.max() - img.min() + 1e-8)
H, W = img.shape

jj, ii = np.meshgrid(np.arange(W), np.arange(H))
cx = pcx * (W - 1)
cy = pcy * (H - 1)

X = (jj - cx) / (pcz * H)
Y = -(ii - cy) / (pcz * H)
Z = np.ones_like(X)

norm = np.sqrt(X**2 + Y**2 + Z**2) + 1e-8
X /= norm
Y /= norm
Z /= norm

mask = img > intensity_threshold
points = np.column_stack([X[mask], Y[mask], Z[mask]])
values = img[mask]

cloud = pv.PolyData(points)
cloud["intensity"] = values

print(f"Using pattern: {pattern_path}")
print(f"Image shape: {H} x {W}")
print(f"Visible points: {len(points)}")

plotter = pv.Plotter()
plotter.add_mesh(
    cloud,
    scalars="intensity",
    cmap="gray",
    point_size=3,
    render_points_as_spheres=True,
)
plotter.add_axes()
plotter.show()
